# Normalización de archivos — Oriente Estéreo
**Python Schools 2026**

Convierte nombres de archivos de audio al formato estándar:
```
titulo_AAAA-MM-DD.ext
titulo_AAAA-MM-DD_epNNN.ext   (si tiene episodio)
titulo_sin_fecha.ext           (si no tiene fecha)
```

## 1. Carga de librerías

In [ ]:
import re
from difflib import get_close_matches
from pathlib import Path

## 2. Carga de datos — Escanear la carpeta

Aquí se leen los archivos reales de la carpeta.
El escáner busca `.mp3` y `.mp4` y devuelve la lista de nombres.

In [ ]:
carpetas = [
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE",
]

In [ ]:
archivos_prueba = [
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 06-06-25.mp3",
    "/content/disco_prueba/PROGRAMAS/BARRIO ADENTRO/BARRIO ADENTRO 17-07-25.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES- AGOSTO 25 DE 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/EN LA RAYA DEPORTES DIC 1 DE 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/EN LA RAYA/En la Raya Deportes- Septiembre 29 de 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DEJAME SER ARTE 03-07-25.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PROGRAMAS/DSA 02-10-25.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA/Arcangel - Andan Diciendo.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/MUSICA/GAMBOA - Fuiste Tu.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/CABEZOTE DSA.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/PISADOR DSA 1.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/PRODUCCION DSA/SEPARADOR DSA 3 FINAL.mp3",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/FONDO DSA.jpg",
    "/content/disco_prueba/PROGRAMAS/DEJAME SER ARTE/FONDO DSA",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 35 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 118.mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 122 2023 (1).mp3",
    "/content/disco_prueba/PROGRAMAS/A PRENDER LA ONDA/Episodio 290 2025.mp3",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - mujeres bolero parte 2.mp3",
    "/content/disco_prueba/PROGRAMAS/PAGANO BOLERO/PAGANO BOLERO - boleros inspirados estaciones dima.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/1 - La historia de la radio y la radio en la historia.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/7 - Comienzos de la radio en Colombia.mp3",
    "/content/disco_prueba/PROGRAMAS/DIANA URIBE/13 - Radio educativa, Radio Sutatenza.mp3",
]

## 3. Definición de funciones

### 3.1 Generación automática del diccionario de meses

In [ ]:
# En lugar de escribir cada mes a mano, definimos una lista base
# con el nombre largo y la abreviación más usada.
# El loop de abajo genera el diccionario solo.

NOMBRES_MESES = [
    ("enero",       "ene"),
    ("febrero",     "feb"),
    ("marzo",       "mar"),
    ("abril",       "abr"),
    ("mayo",        "may" ),
    ("junio",       "jun"),
    ("julio",       "jul"),
    ("agosto",      "ago"),
    ("septiembre",  "sep"),
    ("octubre",     "oct"),
    ("noviembre",   "nov"),
    ("diciembre",   "dic"),
]

# Construir el diccionario automáticamente
MESES = {}
for numero, (nombre_largo, abrev) in enumerate(NOMBRES_MESES, start=1):
    num_str = str(numero).zfill(2)   # 1 -> '01', 9 -> '09'
    MESES[nombre_largo] = num_str     # 'agosto'     -> '08'
    if abrev:
        MESES[abrev] = num_str        # 'ago'        -> '08'
    MESES[nombre_largo[:4]] = num_str # 'agos'       -> '08' (primeras 4 letras)
    MESES[nombre_largo[:3]] = num_str # 'ago' ya existe, pero cubre 'sep','oct'...

# Solo los nombres largos sirven para fuzzy matching (evitar falsos positivos)
MESES_LARGOS = [nombre for nombre, _ in NOMBRES_MESES]

# Palabras que no son parte del título y se eliminan al normalizar
PALABRAS_RUIDO = {
    "de", "del", "la", "el",
    "sabado", "domingo", "lunes", "martes", "miercoles", "jueves", "viernes"
}

# Ver el resultado
print("Meses detectables:", list(MESES.keys()))


### 3.2 Funciones de apoyo

In [ ]:
def separar_extension(nombre):
    """Separa el nombre base de la extensión.
    Ejemplo: 'audio.MP3' -> ('audio', '.mp3')
    """
    if "." in nombre:
        base, ext = nombre.rsplit(".", 1)
        return base, "." + ext.lower()
    return nombre, ""


def limpiar_texto(texto):
    """Pasa el texto a minúsculas y reemplaza
    caracteres especiales por espacio.
    """
    texto = texto.lower()
    for caracter in [",", ".", "(", ")", "!", ";"]:
        texto = texto.replace(caracter, " ")
    return texto


def expandir_anio(anio_str):
    """Convierte un año de 2 dígitos a 4 dígitos.
    Ejemplo: '25' -> '2025', '2025' -> '2025'
    Retorna None si el valor no es un año válido.
    """
    if not anio_str.isdigit():
        return None
    if len(anio_str) == 2:
        return "20" + anio_str
    if len(anio_str) == 4 and 1900 <= int(anio_str) <= 2100:
        return anio_str
    return None


def construir_fecha(dia, mes, anio_str):
    """Arma la fecha en formato AAAA-MM-DD.
    Si día y mes están al revés, los corrige automáticamente.
    Retorna None si los valores son inválidos.
    """
    anio = expandir_anio(anio_str)
    if not anio:
        return None

    d, m = int(dia), int(mes)

    # Verificar si día y mes son válidos
    if not (1 <= m <= 12 and 1 <= d <= 31):
        # Intentar intercambiarlos
        if 1 <= d <= 12 and 1 <= m <= 31:
            d, m = m, d
        else:
            return None

    return f"{anio}-{str(m).zfill(2)}-{str(d).zfill(2)}"


def buscar_mes(token):
    """Busca el número de mes de un token.
    Primero busca coincidencia exacta, luego tolera errores de escritura.
    Ejemplo: 'agosto' -> '08', 'agoto' -> '08' (typo corregido)
    Retorna None si no encuentra el mes.
    """
    # Exacto
    if token in MESES:
        return MESES[token]

    # Fuzzy: acepta palabras con pequeños errores de escritura
    if len(token) >= 4:
        coincidencias = get_close_matches(token, MESES_LARGOS, n=1, cutoff=0.75)
        if coincidencias:
            return MESES[coincidencias[0]]

    return None

### 3.3 Función principal: `detectar_fecha`

In [ ]:
def detectar_fecha(nombre):
    """Detecta la fecha en el nombre de un archivo.

    Soporta estos formatos (en orden de prioridad):
      - AAAA-MM-DD  (ISO)          Archivo 2025-08-15.mp3
      - DD-MM-AA(AA)               BARRIO ADENTRO 06-06-25.mp3
      - DD/MM/AAAA                 Noticias 06/11/2025.mp3
      - DD.MM.AAAA                 Programa 14.03.2026.mp3
      - Mes en español + día + año AGOSTO 25 DE 2025
      - Solo año (último recurso)  Episodio 35 2025.mp3

    Retorna: 'AAAA-MM-DD' | 'AAAA-MM' | 'AAAA' | 'sin_fecha'
    """
    base, _ = separar_extension(nombre)
    texto   = limpiar_texto(base)

    mejor_fecha = None
    mejor_conf  = -1

    # --- Patrones con expresiones regulares ---
    # Cada patrón tiene: regex, cómo extraer (día, mes, año), nivel de confianza
    patrones = [
        # ISO: AAAA-MM-DD  (más confiable porque el año va primero)
        (r"(\d{4})-(\d{1,2})-(\d{1,2})",
         lambda m: construir_fecha(m.group(3), m.group(2), m.group(1)), 95),

        # DD-MM-AA(AA): puede venir pegado a texto, ej: arte17-07-25
        (r"(\d{1,2})-(\d{1,2})-(\d{2,4})",
         lambda m: construir_fecha(m.group(1), m.group(2), m.group(3)), 80),

        # DD/MM/AAAA: separado por barras
        (r"(\d{1,2})/(\d{1,2})/(\d{2,4})",
         lambda m: construir_fecha(m.group(1), m.group(2), m.group(3)), 80),

        # DD.MM.AAAA: separado por puntos (buscar en el nombre original)
        (r"(\d{1,2})\.(\d{1,2})\.(\d{2,4})",
         lambda m: construir_fecha(m.group(1), m.group(2), m.group(3)), 80),
    ]

    # Probar cada patrón sobre el texto limpio y sobre el nombre original
    for fuente in [texto, base.lower()]:
        for patron, extraer, confianza in patrones:
            for m in re.finditer(patron, fuente):
                fecha = extraer(m)
                if fecha and confianza > mejor_conf:
                    mejor_fecha = fecha
                    mejor_conf  = confianza

    # --- Patrón: mes en español (con tolerancia a errores de escritura) ---
    # Solo se activa si no se encontró una fecha más confiable
    if mejor_conf < 85:
        tokens = texto.replace("-", " ").split()

        for i, token in enumerate(tokens):
            num_mes = buscar_mes(token)
            if not num_mes:
                continue

            dia = None
            anio = None

            # Buscar día y año cerca del mes (ventana de ±4 tokens)
            for j, candidato in enumerate(tokens):
                if not candidato.isdigit():
                    continue
                valor = int(candidato)
                if 1900 <= valor <= 2100:
                    anio = str(valor)
                elif 1 <= valor <= 31 and j != i and abs(j - i) <= 4 and not dia:
                    dia = str(valor).zfill(2)

            if anio and dia:
                fecha = construir_fecha(dia, num_mes, anio)
                if fecha:
                    mejor_fecha = fecha
                    mejor_conf  = 85
            elif anio:
                anio_completo = expandir_anio(anio)
                if anio_completo:
                    mejor_fecha = f"{anio_completo}-{num_mes}"
                    mejor_conf  = 70
            break  # Ya encontramos el mes, no seguir buscando

    # --- Último recurso: solo año de 4 dígitos ---
    if not mejor_fecha:
        m = re.search(r"\b((?:19|20)\d{2})\b", texto)
        if m:
            mejor_fecha = m.group(1)

    return mejor_fecha or "sin_fecha"

### 3.4 Función: `detectar_episodio`

In [ ]:
def detectar_episodio(nombre):
    """Detecta el número de episodio en el nombre de un archivo.

    Reconoce estos estilos:
      - 'Episodio 35 ...'        -> ep035
      - 'PROGRAMA 286 ...'       -> ep286
      - 'PROG. 293 ...'          -> ep293
      - '7 - Título del audio'   -> ep007

    Retorna: 'epNNN' o None si no hay episodio.
    """
    base, _ = separar_extension(nombre)
    texto   = limpiar_texto(base)
    tokens  = texto.split()

    # Buscar palabras clave seguidas de un número
    palabras_clave = ["episodio", "programa", "prog"]
    for palabra in palabras_clave:
        if palabra in tokens:
            i = tokens.index(palabra)
            if i + 1 < len(tokens):
                # El número puede venir solo '35' o pegado '34-2025'
                m = re.match(r"(\d+)", tokens[i + 1])
                if m:
                    numero = str(int(m.group(1))).zfill(3)
                    return "ep" + numero

    # Buscar estilo 'N - Título'
    m = re.match(r"^(\d+)\s*-\s*\S", base.strip())
    if m:
        numero = str(int(m.group(1))).zfill(3)
        return "ep" + numero

    return None

### 3.5 Función: `normalizar_nombre`

In [ ]:
def limpiar_titulo(nombre, fecha, episodio):
    """Extrae el título limpio: quita la fecha, el episodio
    y palabras de relleno del nombre.
    """
    base, _ = separar_extension(nombre)
    texto   = limpiar_texto(base)

    # Quitar la fecha del texto
    if fecha != "sin_fecha":
        # Quitar el año de 4 dígitos
        anio = fecha[:4]
        texto = re.sub(r"\b" + anio + r"\b", " ", texto)
        # Quitar mes y día si la fecha es completa
        if len(fecha) == 10:  # AAAA-MM-DD
            mes = str(int(fecha[5:7]))
            dia = str(int(fecha[8:10]))
            # Quitar todas las formas de mes (nombre y número)
            for nombre_mes, num_mes in MESES.items():
                if num_mes == fecha[5:7]:
                    texto = re.sub(r"\b" + nombre_mes + r"\b", " ", texto)
            texto = re.sub(r"\b" + mes + r"\b", " ", texto)
            texto = re.sub(r"\b" + dia + r"\b", " ", texto)

    # Quitar números de episodio (ej: '293', '35')
    if episodio:
        numero = str(int(episodio[2:])).zfill(1)  # 'ep293' -> '293'
        texto = re.sub(r"\b" + numero + r"\b", " ", texto)

    # Quitar palabras de ruido
    tokens = texto.split()
    partes = []
    for token in tokens:
        if token in PALABRAS_RUIDO:
            continue
        if token == "-":
            continue
        # Quitar números sueltos pequeños (días o meses que quedaron)
        if token.isdigit() and 1 <= int(token) <= 31:
            continue
        # Convertir el token a formato seguro para nombre de archivo
        token_limpio = re.sub(r"[^a-z0-9]", "_", token).strip("_")
        if token_limpio:
            partes.append(token_limpio)

    return "_".join(partes)


def normalizar_nombre(nombre):
    """Convierte un nombre de archivo al formato estándar.

    Formato de salida:
      titulo_AAAA-MM-DD.ext
      titulo_AAAA-MM-DD_epNNN.ext   (si tiene episodio)
      titulo_sin_fecha.ext           (si no tiene fecha)
    """
    _, ext     = separar_extension(nombre)
    fecha      = detectar_fecha(nombre)
    episodio   = detectar_episodio(nombre)
    titulo     = limpiar_titulo(nombre, fecha, episodio)

    if episodio:
        return f"{titulo}_{fecha}_{episodio}{ext}"
    return f"{titulo}_{fecha}{ext}"

## 4. Aplicar a los archivos reales

Aquí se procesan los archivos que escaneó la celda 2.
No hay nombres predeterminados — trabaja con lo que encontró en la carpeta.

In [ ]:
# Mostrar el resultado de la normalización para cada archivo encontrado
print(f"{'ANTES':<55} {'DESPUÉS'}")
print("-" * 110)

for nombre in archivos:
    nuevo = normalizar_nombre(nombre)
    print(f"{nombre:<55} {nuevo}")

## 5. Verificación — Probar con tus propios archivos

Escribe en `mis_archivos` los nombres que quieres probar.
Pueden ser nombres reales de tu carpeta o cualquier nombre inventado.
La función detecta la fecha y el episodio **automáticamente** leyendo el nombre.
Los que no tienen fecha aparecen como `sin_fecha`.

In [ ]:
# ↓ Escribe aquí los nombres que quieres probar
# Pueden ser nombres reales de tu carpeta o nombres de prueba
mis_archivos = [
    # "BARRIO ADENTRO 06-06-25.mp3",
    # "EN LA RAYA AGOSTO 25 DE 2025.mp3",
    # Agrega los tuyos aquí...
]

# Si no escribiste nada arriba, usa los archivos escaneados automáticamente
if not mis_archivos:
    mis_archivos = archivos

# Mostrar resultados
print(f"{'FECHA':<14} {'EPISODIO':<10} {'NOMBRE'}")
print("-" * 80)

for nombre in mis_archivos:
    fecha    = detectar_fecha(nombre)
    episodio = detectar_episodio(nombre) or "sin episodio"
    print(f"{fecha:<14} {episodio:<12} {nombre}")

print()
print("Resultado normalizado:")
print("-" * 80)
for nombre in mis_archivos:
    print(f"  ANTES  : {nombre}")
    print(f"  DESPUÉS: {normalizar_nombre(nombre)}")
    print()
